<a href="https://colab.research.google.com/github/swirty/csc591-Hybrid-DL-ML-Classifier/blob/main/LiveDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from tqdm.notebook import tqdm
from PIL import Image
import torchvision.transforms as transforms

import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import torch.nn as nn
import torch

from sklearn.svm import SVC
import joblib

dir = '/content/drive/MyDrive/Colab-Hybrid-Ransomware/test'
filenames = ['/content/drive/MyDrive/Colab-Hybrid-Ransomware/at.exe']

nn_model_save_path = f'/content/drive/MyDrive/Colab-Hybrid-Ransomware/CNN/CNN_model_full.pth'

model_save_path = f'/content/drive/MyDrive/Colab-Hybrid-Ransomware/CNN/CNN_rf_model_full.joblib'

In [ ]:
def load_file_raw(filepath):
    # a. Check if the file exists
    if not os.path.exists(filepath):
        print(f"Error: File '{filepath}' not found. Please ensure the path is correct.")
        return None, None # Return None, None if file not found

    # b. Load the specified file
    #print(f"\nLoading file from: {filepath}")
    try:
        with open(filepath, 'rb') as f:
            file_data = f.read()
            filename = os.path.basename(filepath) # Get the filename from the full path
            return file_data, filename
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return None, None # Return None, None if reading fails

def get_size(data_length, width=None):
    if width is None:
        size = data_length
        if (size < 10240):
            width = 32
        elif (10240 <= size <= 10240 * 3):
            width = 64
        elif (10240 * 3 <= size <= 10240 * 6):
            width = 128
        elif (10240 * 6 <= size <= 10240 * 10):
            width = 256
        elif (10240 * 10 <= size <= 10240 * 20):
            width = 384
        elif (10240 * 20 <= size <= 10240 * 50):
            width = 512
        elif (10240 * 50 <= size <= 10240 * 100):
            width = 768
        else:
            width = 1024

        height = int(size / width) + 1
    else:
        width  = int(math.sqrt(data_length)) + 1
        height = width

    return (width, height)

def make_image_data(data, size, image_type):
    try:
        image = Image.new(image_type, size)
        image.putdata(data)
        return image
    except Exception as err:
        print(err)

def convert_bytes_to_greyscale_image(raw_bytes):
    byte_values = list(raw_bytes)
    size = get_size(len(byte_values))
    return make_image_data(byte_values, size, 'L')


Image.MAX_IMAGE_PIXELS = None
target_size = (256, 256)

def repeat_channel_three_times(x):
    if x.size(0) == 1:
        return x.repeat(3, 1, 1)
    return x

transform = transforms.Compose([
    # Convert the PIL Image to a PyTorch tensor using transforms.ToTensor().
    transforms.ToTensor(),
    # Use the named function to convert single-channel (greyscale)
    # images to 3-channel images by repeating the single channel three times
    transforms.Lambda(repeat_channel_three_times),
    # Resize
    transforms.Resize(target_size, antialias=True)
])

def path_to_tensor(path):
  raw, name = load_file_raw(path)
  image = convert_bytes_to_greyscale_image(raw)
  return transform(image)

In [ ]:
tensor_list = []
path_list = []
for filename in tqdm(os.listdir(dir), desc="Processing files"):
    if filename.lower().endswith('.exe'):
        filepath = os.path.join(dir, filename)
        tensor_list.append(path_to_tensor(filepath))
        path_list.append(filepath)

print(f"Created a list of {len(tensor_list)} tensors.")

In [ ]:
# Model Definition For CNN

class ConvNet(nn.Module):
    def __init__(self, num_classes=110):
        super(ConvNet, self).__init__()

        self.features = torch.nn.Sequential(

            nn.Conv2d(3, 224, kernel_size=11, stride=2, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(224, 192, kernel_size=5, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))

        # Define the part of the classifier that produces the 4096-dim feature vector
        self.feature_extractor_head = torch.nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU()
        )

        # Define the rest of the classifier for final predictions
        self.classification_head = torch.nn.Sequential(
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x, extract_features=False):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1) # This is the 9216-dim vector

        # Apply the feature extraction head
        extracted_4096_features = self.feature_extractor_head(x)

        if extract_features:
            return extracted_4096_features # Return the 4096-dimensional features
        else:
            # Continue with the classification head for normal prediction
            out = self.classification_head(extracted_4096_features)
            return out

print("ConvNet model defined.")

In [ ]:
conv_model = ConvNet(num_classes=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
conv_model.to(device)

# Load the saved state dictionary
conv_model.load_state_dict(torch.load(nn_model_save_path, map_location=device))
conv_model.eval() # Set the model to evaluation mode after loading

print(f"CNN model loaded successfully from {nn_model_save_path}")

In [ ]:
extracted_features = []
with torch.no_grad():
    for i, tensor in enumerate(tqdm(tensor_list, desc="Extracting features")):
        # Add a batch dimension (unsqueeze(0)) and move to device
        features = conv_model(tensor.unsqueeze(0).to(device), extract_features=True)
        extracted_features.append(features)

In [ ]:
import matplotlib.pyplot as plt

# Get the first feature vector (already a NumPy array from previous steps)
first_feature_vector = extracted_features[400].squeeze().cpu().numpy()

# Create a range for the x-axis (indices of the features)
x_pos = np.arange(len(first_feature_vector))

# Create the bar plot
plt.figure(figsize=(15, 5)) # Adjust figure size for better readability
plt.bar(x_pos, first_feature_vector, width=1.0) # Use width=1.0 for contiguous bars
plt.xlabel("Feature Index")
plt.ylabel("Feature Value")
plt.title("Elements of the Feature Vector")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
model = joblib.load(model_save_path)
print(f"SVM model loaded successfully from {model_save_path}")

In [ ]:
extracted_features_tensor = torch.cat(extracted_features, dim=0)
extracted_features_np = extracted_features_tensor.cpu().numpy()

predictions = []
for feature_vector in tqdm(extracted_features_np, desc="Predicting"):
    predictions.append(model.predict(feature_vector.reshape(1, -1))[0])
predictions = np.array(predictions)

In [ ]:
for i, filepath in enumerate(path_list):
    print(f"File: {filepath.split('/')[-1]}, Prediction: {predictions[i]}")